# Training Demo

This notebook demonstrates how to train the Neural Receiver model.

Topics covered:
1. Model architecture overview
2. Dataset preparation
3. Training the multi-task model
4. Monitoring training progress
5. Visualizing results

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

from src.data import SignalDataset, ModulationType
from src.models import NeuralReceiver, MultiTaskLoss
from src.training import Trainer
from src.utils import plot_training_history

## 1. Model Architecture Overview

In [ ]:
# Create model
model = NeuralReceiver(
    input_channels=2,      # IQ data (I and Q)
    base_channels=64,      # Base feature channels
    num_blocks=4,          # Number of residual blocks
    num_classes=12,        # Number of modulation types
    dropout=0.2
)

# Print model summary
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Dataset Preparation

In [ ]:
# Training configuration
config = {
    'sequence_length': 1024,
    'train_samples': 5000,  # Reduced for demo (use 20000+ for real training)
    'val_samples': 1000,
    'snr_range': (-10, 0),  # Weak signals
    'no_signal_prob': 0.2,
    'batch_size': 32,
    'num_workers': 4,
    'seed': 42
}

# Create datasets
print("Creating datasets...")
train_dataset = SignalDataset(
    n_samples=config['train_samples'],
    sequence_length=config['sequence_length'],
    snr_range=config['snr_range'],
    no_signal_prob=config['no_signal_prob'],
    seed=config['seed'],
    pregenerate=False
)

val_dataset = SignalDataset(
    n_samples=config['val_samples'],
    sequence_length=config['sequence_length'],
    snr_range=config['snr_range'],
    no_signal_prob=config['no_signal_prob'],
    seed=config['seed'] + 1000,
    pregenerate=False
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=SignalDataset.collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    collate_fn=SignalDataset.collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 3. Setup Training Components

In [ ]:
# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Multi-task loss function
criterion = MultiTaskLoss(
    detection_weight=1.0,
    regression_weight=1.0,
    classification_weight=1.0,
    use_uncertainty_weighting=False  # Set to True for learned weighting
)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-6
)

## 4. Train the Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    output_dir='../experiments',
    experiment_name='demo_training',
    use_tensorboard=True
)

# Train for a few epochs (increase for better performance)
num_epochs = 10  # Use 50-100 for real training

print(f"Training for {num_epochs} epochs...")
trainer.train(num_epochs=num_epochs, save_every=5)

## 5. Visualize Training Progress

In [ ]:
# Plot training history
plot_training_history(trainer.training_history)

## 6. Test Inference

In [ ]:
# Get a test sample
iq_data, labels = val_dataset[0]
iq_batch = iq_data.unsqueeze(0).to(device)

# Run inference
model.eval()
with torch.no_grad():
    predictions = model.predict(iq_batch)

# Print results
class_names = ModulationType.get_class_names()

print("Ground Truth:")
print(f"  Signal present: {labels['signal_present'].item()}")
print(f"  Modulation: {class_names[labels['modulation_class'].item()]}")
print(f"  SNR: {labels['snr_db'].item():.2f} dB")
print(f"  Center freq: {labels['center_freq'].item():.4f}")
print(f"  Bandwidth: {labels['bandwidth'].item():.4f}")

print("\nPredictions:")
print(f"  Signal detected: {predictions['signal_detected'][0].item()}")
print(f"  Detection prob: {predictions['detection_prob'][0].item():.4f}")
print(f"  Modulation: {class_names[predictions['modulation_class'][0].item()]}")
print(f"  SNR: {predictions['snr_db'][0].item():.2f} dB")
print(f"  Center freq: {predictions['center_freq'][0].item():.4f}")
print(f"  Bandwidth: {predictions['bandwidth'][0].item():.4f}")

## 7. View TensorBoard Logs (Optional)

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Launch TensorBoard
%tensorboard --logdir ../experiments/demo_training/logs

## Summary

In this notebook, we:
1. Created a multi-task neural receiver model
2. Prepared training and validation datasets
3. Trained the model with multi-task loss
4. Monitored training progress
5. Tested inference on sample data

Next steps:
- Train for more epochs for better performance
- Evaluate on test set (see `03_evaluation_demo.ipynb`)
- Tune hyperparameters (learning rate, architecture, etc.)